# 05 — Day 5 Feature Extraction: Speed-Perturbed HuBERT (Layer 12)

**Standalone notebook** (same reasoning as Day 4): the Day 3 layer-sweep notebook was already saved, so this doesn't touch it — it only extracts the two *new* things Day 5 needs.

**What this does:** for each of the 280 recordings, create two speed-perturbed copies of the audio — **10% faster (speed 1.1)** and **10% slower (speed 0.9)** — and run each through the confirmed winning embedding, **HuBERT-base layer 12 only** (not all 13 layers — Day 3 already picked the winner, no need to sweep again). The unperturbed (speed 1.0 / original) embeddings are **not** re-extracted here — they're already cached in Day 3's `hubert_base_all_layers.parquet`, and reusing them keeps this augmented setup perfectly consistent with the confirmed 0.611 baseline rather than introducing tiny extraction-run-to-run differences.

**How speed perturbation works:** resample the trimmed audio to a different rate, then feed it to the model as if it were still 16kHz. This is the classic technique (Ko et al. 2015) — it changes both tempo *and* pitch slightly together (unlike pitch-preserving time-stretching), mimicking someone talking faster or slower.

**The real trade-off, stated precisely (not just "pitch gets blurred"):** speaking rate is a well-documented genuine fatigue marker in the speech literature (slower articulation, more/longer pauses, and more total speech time all correlate with fatigue in prior research) — so this isn't a safely-inert augmentation. By labeling a recording's original, 10%-slower, and 10%-faster copies identically, this technique explicitly trains the classifier toward **tempo-invariance**. If part of HuBERT layer 12's confirmed 0.611 result comes from genuinely picking up on tempo differences, this augmentation works directly against that signal, not just against noise. That's exactly what Day 5's evaluation checks empirically — see `05_evaluate_hubert_augmented.ipynb`'s closing section for how to read a result in either direction.

**Inputs needed:** just the `Fatigue_data` dataset (`project_data/`) — no need to attach Day 3's output here; that gets combined with this notebook's output in the evaluation notebook.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

DATA_DIR = next(Path("/kaggle/input").rglob("project_data"))
AUDIO_DIR = DATA_DIR / "audio"
LABELS_CSV = DATA_DIR / "labels.csv"

OUT_DIR = Path("/kaggle/working/features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
LAYER = 12
SPEED_FACTORS = [0.9, 1.1]  # 1.0 (original) reused from Day 3's cache, not regenerated

labels = pd.read_csv(LABELS_CSV)
missing = [f for f in labels["filename"] if not (AUDIO_DIR / f).exists()]
assert not missing, f"{len(missing)} filenames missing audio files: {missing[:5]}"
assert len(labels) == 280, f"expected 280 rows, got {len(labels)}"

print(f"DATA_DIR  = {DATA_DIR}")
print(f"AUDIO_DIR = {AUDIO_DIR}")
print(f"Loaded {len(labels)} labeled recordings.")

## 1. Load HuBERT (layer 12 only this time)

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from transformers import Wav2Vec2FeatureExtractor, HubertModel

HUBERT_MODEL_NAME = "facebook/hubert-base-ls960"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(HUBERT_MODEL_NAME)
model = HubertModel.from_pretrained(HUBERT_MODEL_NAME).to(device)
model.eval()  # frozen, same as every other extraction notebook in this project

## 2. Speed Perturbation & Extraction

In [ ]:
def speed_perturb(y: np.ndarray, sr: int, speed: float) -> np.ndarray:
    """Classic resample-based speed perturbation (Ko et al. 2015).
    speed > 1.0 -> faster & higher-pitched; speed < 1.0 -> slower & lower-pitched.
    Resample to sr/speed, then the result is treated as if still sampled at sr --
    that mismatch is exactly what produces the speed change."""
    return librosa.resample(y, orig_sr=sr, target_sr=int(sr / speed))


def extract_hubert_layer12(y: np.ndarray) -> np.ndarray:
    """Frozen HuBERT forward pass -> layer 12 hidden states -> mean+std pool.
    Same pooling as every other extraction notebook, just fixed to one layer."""
    inputs = feature_extractor(y, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = model(input_values, output_hidden_states=True)
        hidden = outputs.hidden_states[LAYER].squeeze(0)  # (T, 768)

    mean_pool = hidden.mean(dim=0)
    std_pool = hidden.std(dim=0)
    return torch.cat([mean_pool, std_pool]).cpu().numpy()  # (1536,)

In [ ]:
from tqdm.auto import tqdm

rows = []
for filename in tqdm(labels["filename"], desc="Extracting speed-perturbed embeddings"):
    y, _ = librosa.load(AUDIO_DIR / filename, sr=SAMPLE_RATE, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=25)  # trim BEFORE perturbing, same as every other notebook

    for speed in SPEED_FACTORS:
        y_perturbed = speed_perturb(y_trimmed, SAMPLE_RATE, speed)
        emb = extract_hubert_layer12(y_perturbed)
        rows.append({
            "source_filename": filename,
            "speed_factor": speed,
            **{f"emb_{i}": v for i, v in enumerate(emb)},
        })

augmented_df = pd.DataFrame(rows)
print(augmented_df.shape)  # expect (560, 2 + 1536)
augmented_df.head()

## 3. Cache & Verify

In [ ]:
out_path = OUT_DIR / "hubert_layer12_speed_augmented.parquet"
augmented_df.to_parquet(out_path, index=False)

assert len(augmented_df) == 280 * len(SPEED_FACTORS), \
    f"expected {280 * len(SPEED_FACTORS)} rows, got {len(augmented_df)}"
n_emb_cols = sum(c.startswith("emb_") for c in augmented_df.columns)
assert n_emb_cols == 1536, f"expected 1536 embedding dims, got {n_emb_cols}"
assert set(augmented_df["source_filename"]) == set(labels["filename"]), "filename mismatch vs labels.csv"
assert set(augmented_df["speed_factor"].unique()) == set(SPEED_FACTORS)

print(f"Saved {out_path}")
print(f"Shape: {augmented_df.shape} (280 recordings x {len(SPEED_FACTORS)} speed variants x 1536 dims)")
print("All verification checks passed.")

## Next steps

1. **Save this notebook's version.**
2. Open `05_evaluate_hubert_augmented.ipynb`, attach **both** this notebook's output (`hubert_layer12_speed_augmented.parquet`) **and** Day 3's `01_features.ipynb` output (`hubert_base_all_layers.parquet`, for the original/unperturbed layer-12 embeddings), plus the `Fatigue_data` dataset. It combines all three speed variants (0.9 / 1.0 / 1.1) into one training-augmented nested-CV evaluation, and compares against the confirmed baseline (AUC 0.611, CI [0.551, 0.683], p = 0.01).